# Imports

In [ ]:
# Standard imports
import tables_io
import hpmcm
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits


# setup for one tract

In [ ]:
# Set the tract
tract = 8006
# This will need to change with tract
ref_dir=(340.5000000000, -9.6694214876)
our_geom = hpmcm.shear_utils.ShearCellGeometry(
    ref_dir=ref_dir,
)

In [ ]:
test_only = True

rubin_shear_match_files = [
    f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq',    
]
rubin_shear_match_ids = list(range(len(rubin_shear_match_files)))

all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_joined.pq',
    f'{tract}/rubin_md_match_object_joined.pq',
]
all_ids = list(range(len(all_files)))

### Get the coadd for one cell

In [ ]:
file_template = "8006_deep_coadd/deep_coadd_8006_{patch}_i_lsst_cells_v2_LSSTCam_runs_DRP_DP2_v30_0_8_DM-55060_stage3_20260609T040412Z.fits"
def get_patch_coadd(patch_x, patch_y):
    patch_id = int(10*patch_y + patch_x)
    fname = file_template.format(patch=patch_id)
    coadd = fits.open(fname)
    fits_image = coadd[1]
    corner = (fits_image.header['CRVAL1A'], fits_image.header['CRVAL2A'])
    patch_size = (fits_image.header['NAXIS1'], fits_image.header['NAXIS2'])
    extent = (corner[0], corner[0] + patch_size[0], corner[1], corner[1] + patch_size[1]) 
    return coadd[1], extent

def get_cell_from_patch(patch_coadd, patch_extent, cell_id_x, cell_id_y, buffer=25):
    offset_x = cell_id_x * 150
    offset_y = cell_id_y * 150
    low_corner_x = offset_x - buffer
    low_corner_y = offset_y - buffer
    hi_corner_x = low_corner_x + 150 + 2*buffer
    hi_corner_y = low_corner_y + 150 + 2*buffer
    extent = (low_corner_x, hi_corner_x, low_corner_y, hi_corner_y)
    global_extent = (
        extent[0]+ patch_extent[0], 
        extent[1]+ patch_extent[0],                 
        extent[2]+ patch_extent[2], 
        extent[3]+ patch_extent[2],         
    )        
    return patch_coadd.data[low_corner_y:hi_corner_y, low_corner_x:hi_corner_x], global_extent

def get_cell_coadd(cell_id):    
    cell_global_x = cell_id//201
    cell_global_y = cell_id%201
    patch_idx_x = (cell_global_x -1) // 20
    patch_idx_y = (cell_global_y -1) // 20
    cell_idx_x = (cell_global_x - 1) % 20 + 1
    cell_idx_y = (cell_global_y - 1) % 20 + 1
    patch_coadd, patch_extent = get_patch_coadd(patch_idx_x, patch_idx_y)
    cell_coadd, cell_extent = get_cell_from_patch(patch_coadd, patch_extent, cell_idx_x, cell_idx_y)
    return cell_coadd, cell_extent, (cell_global_x, cell_global_y, patch_idx_x, patch_idx_y, cell_idx_x, cell_idx_y)

In [ ]:
cc, extent, ids = get_cell_coadd(20200)

_ = plt.imshow(cc.data, norm='log', cmap='gray_r', origin='lower', vmin=1, vmax=1e3, extent=extent)
_ = plt.colorbar()


## Run the matching for shear object catalogs

In [ ]:
rubin_shear_match = hpmcm.shear_match.ShearMatch.createShearMatch(
    geometry=our_geom,
    pixel_match_scale=2,
    pixel_r2_cut=16,
    deshear=-0.01,
    shear_names=['ns', '1m', '1p', '2m', '2p'],        
)
rubin_shear_match.reduceData(rubin_shear_match_files, rubin_shear_match_ids)
if test_only:
    rubin_shear_match.analysisLoop(range(100, 101), range(100, 101))
else:
    rubin_shear_match.analysisLoop()
rubin_stats = rubin_shear_match.extractStats()
rubin_stats.update(**rubin_shear_match.extractShearStats())


### Grab the results for one cell

In [ ]:
cell_data = rubin_shear_match.cell_dict[20200]
extent = (
    cell_data.corner[0],
    cell_data.corner[0]+cell_data.size[0], 
    cell_data.corner[1],
    cell_data.corner[1]+cell_data.size[1], 
)

#### Show the source maps

In [ ]:
markers = ['x', '<', '>', '^', 'v']
for marker, cat in zip(markers, cell_data.data):
    _ = plt.scatter(cat.x_pix, cat.y_pix, marker=marker, s=5)
_ = plt.xlim(cell_data.corner[0], cell_data.corner[0]+cell_data.size[0])
_ = plt.ylim(cell_data.corner[1], cell_data.corner[1]+cell_data.size[1])

### show the source counts map and the associated cluster table

In [ ]:
_ = plt.imshow(cell_data.countsMap().T, origin='lower', extent=extent, cmap='gray_r')
_ = plt.colorbar()

In [ ]:
rubin_stats['cluster_stats']

#### Show the clusters

In [ ]:
cluster_id = list(cell_data.cluster_dict.keys())[4]
cluster_data = cell_data.cluster_dict[cluster_id]
img = cell_data.countsMap()
_ = hpmcm.viz_utils.showCluster(
    img,
    cluster_data,
    cell_data,
)

In [ ]:
shear_rubin_obj_stats = rubin_stats['object_stats']
shear_rubin_obj_assoc = rubin_stats['object_assoc']
joined_shear_obj_full = hpmcm.output_tables.buildJoinedObjectTable(
    shear_rubin_obj_stats,
    shear_rubin_obj_assoc,
    rubin_shear_match_files,
    rubin_shear_match_ids,
)

In [ ]:
joined_shear_obj_full

In [ ]:
cell = joined_shear_obj_full.cell_idx_x_0 + joined_shear_obj_full.cell_idx_y_0*201
cc, extent, ids = get_cell_coadd(cell[0])

In [ ]:
cell

In [ ]:
_ = plt.imshow(cc.data, norm='log', cmap='gray_r', origin='lower', vmin=1, vmax=1e3, extent=extent)
_ = plt.colorbar()

markers = ['x', '<', '>', '^', 'v']
for marker, cat in zip(markers, cell_data.data):
    _ = plt.scatter(cat.x_pix, cat.y_pix, marker=marker, s=10)
_ = plt.xlim(extent[0], extent[1])
_ = plt.ylim(extent[2], extent[3])


In [ ]:
pixel_scale = 0.2
n_pix_region = 30000
cell_size = 150
region_size_deg = n_pix_region*pixel_scale/3600
wcs_match = hpmcm.WcsMatch.create(
    ref_dir=ref_dir,
    region_size=(region_size_deg, region_size_deg),
    pixel_size=pixel_scale/3600,
    cell_size=cell_size,
    cell_buffer=5,
    cell_max_object=10000,
    pixel_match_scale=4,
    pixel_r2_cut=16,                
)
    
wcs_match.reduceData(all_files, all_ids)
if test_only:
    wcs_match.analysisLoop(range(100, 101), range(100, 101))
else:
    wcs_match.analysisLoop()
stats = wcs_match.extractStats()

In [ ]:
cell_data_obj = wcs_match.cell_dict[20200]


In [ ]:
_ = plt.imshow(cc.data, norm='log', cmap='gray_r', origin='lower', vmin=1, vmax=1e3, extent=extent)
_ = plt.colorbar()

markers = ['.', 'o', '+', 'x']
for marker, cat in zip(markers, cell_data_obj.data):
    _ = plt.scatter(cat.x_pix, cat.y_pix, marker=marker)
_ = plt.xlim(extent[0], extent[1])
_ = plt.ylim(extent[2], extent[3])


In [ ]:
shear_rubin_in = tables_io.read(f'{tract}/shear_object_{tract}.parq')
shear_rubin_ns = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq')
shear_rubin_1m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq')
shear_rubin_1p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq')
shear_rubin_2m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq')
shear_rubin_2p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq')    
shear_rubin_obj_stats = tables_io.read(f'{tract}/rubin_md_match_object_stats.pq')
shear_rubin_obj_assoc = tables_io.read(f'{tract}/rubin_md_match_object_assoc.pq')
input_files = [
    f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq',
]    
catalog_ids = [0, 1, 2, 3, 4]
joined_shear_obj_full = hpmcm.output_tables.buildJoinedObjectTable(
    shear_rubin_obj_stats,
    shear_rubin_obj_assoc,
    input_files,
    catalog_ids,
)
cell = 7066

joined_shear_obj_cell = joined_shear_obj_full[joined_shear_obj_full.cell_idx == cell]
good_mask = joined_shear_obj_cell.catalog_mask == 31
bad_mask = joined_shear_obj_cell.catalog_mask != 31

joined_shear_obj_cell_good = joined_shear_obj_cell[good_mask]
joined_shear_obj_cell_bad = joined_shear_obj_cell[bad_mask]
cc, extent, ids = get_cell_coadd(cell)

In [ ]:

_ = plt.imshow(cc.data, norm='log', cmap='gray_r', origin='lower', vmin=1, vmax=1e3, extent=extent)
_ = plt.colorbar()
_ = plt.scatter(joined_shear_obj_cell_good.x_pix, joined_shear_obj_cell_good.y_pix)
_ = plt.scatter(joined_shear_obj_cell_bad.x_pix, joined_shear_obj_cell_bad.y_pix)
_ = plt.xlim(extent[0], extent[1])
_ = plt.ylim(extent[2], extent[3])
#_ = plt.scatter(shear_rubin_ns.x_pix, shear_rubin_ns.y_pix, marker='+')
#_ = plt.scatter(shear_rubin_1m.x_pix, shear_rubin_1m.y_pix, marker='<')
#_ = plt.scatter(shear_rubin_1p.x_pix, shear_rubin_1p.y_pix, marker='>')
#_ = plt.scatter(shear_rubin_2m.x_pix, shear_rubin_2m.y_pix, marker='^')
#_ = plt.scatter(shear_rubin_2m.x_pix, shear_rubin_2m.y_pix, marker='v')
_ = plt.scatter(shear_rubin_in.x, shear_rubin_in.y, marker='+')
